# StoneSense-AI: CT Multi-Model Experimentation — Comprehensive Comparison

### 📋 Overview
This notebook conducts the final **head-to-head comparison** across all four CT model families:
1. **ResNet18** (Existing transfer learning baseline evaluated via `common_eval.py`)
   * *Checkpoint location requirement:* `dl/models/kidney_resnet18.pth`
2. **DINOv3 / DINOv2** (Vision foundation model + champion classifier head)
3. **YOLO26-cls** (Ultralytics end-to-end vision model)
4. **QKNN** (PennyLane quantum kernel nearest neighbor classifier)

---
### 📊 Key Analysis Objectives
* Compares Slice Accuracy, Slice Macro F1, Cluster Macro F1, Stone Recall, and Tumor Recall with **95% Cluster-Bootstrap CIs**.
* Conducts non-parametric pairwise hypothesis testing to determine whether performance differences between models fall **within or outside the 95% Confidence Intervals**.
* Evaluates pooled (validation + test) performance metrics (noting that the test set is no longer untouched).
* Generates comparative charts (macro F1 comparison with error bars, confusion matrix grids).
* Exports the definitive `dl/models/ct/comparison.json` report.

---
### 1. Global Configuration & Smoke Test Settings
* **What it does:** Sets filepaths, loads `common_eval.py`, prepares output directory, and sets `SMOKE_TEST` flag.
* **Expected runtime:** < 1 second.
* **Expected output:** Path confirmation.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import pandas as pd
import numpy as np

WORKSPACE_ROOT = Path("/content/StoneSense-AI") if "google.colab" in sys.modules else Path(".").resolve()
sys.path.insert(0, (WORKSPACE_ROOT / "dl" / "notebooks").as_posix())

SMOKE_TEST = False  # Set to True for quick check (20 bootstrap resamples)
BOOTSTRAP_RESAMPLES = 20 if SMOKE_TEST else 1000

MODELS_DIR = WORKSPACE_ROOT / "dl" / "models" / "ct"
MANIFEST_PATH = WORKSPACE_ROOT / "dl" / "outputs" / "reports" / "grouped_split_manifest.csv"
COMPARISON_OUT = MODELS_DIR / "comparison.json"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace Root: {WORKSPACE_ROOT.as_posix()}")
print(f"CT Models Directory: {MODELS_DIR.as_posix()}")
print(f"Smoke Test Mode: {SMOKE_TEST}")

---
### 2. Check Cell: Verify ResNet18 Baseline Checkpoint & Candidate Artifacts
* **What it does:** Evaluates the existing ResNet18 checkpoint on `validation` and `test` splits using `common_eval.py` if not already generated, and verifies that `dinov3`, `yolo26`, and `qknn` artifacts exist.
* **Expected runtime:** ~10-20 seconds.
* **Expected output:** Status of artifacts for all four model families.

In [ ]:
import torch
from common_eval import generate_full_evaluation, save_contract_artifacts, CLASS_NAMES

# Required placement for ResNet18 checkpoint: dl/models/kidney_resnet18.pth
resnet18_ckpt = WORKSPACE_ROOT / "dl" / "models" / "kidney_resnet18.pth"
resnet18_dir = MODELS_DIR / "resnet18"
resnet18_metrics_file = resnet18_dir / "metrics.json"

if not resnet18_metrics_file.exists() and resnet18_ckpt.exists():
    print(f"Found ResNet18 checkpoint at {resnet18_ckpt.as_posix()}. Evaluating with common_eval standard...")
    from torchvision import transforms
    from torchvision.models import resnet18
    from PIL import Image
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = resnet18(num_classes=4)
    ckpt = torch.load(resnet18_ckpt, map_location=device)
    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state)
    model.to(device).eval()
    
    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    df_manifest = pd.read_csv(MANIFEST_PATH)
    
    def eval_rn18(split):
        sub = df_manifest[df_manifest["destination_split"] == split].reset_index(drop=True)
        if SMOKE_TEST:
            sub = sub.head(50)
        probs_list = []
        for _, r in sub.iterrows():
            p = WORKSPACE_ROOT / "dl" / "processed_grouped" / split / r["class"] / r["filename"]
            with Image.open(p) as img:
                t = tf(img.convert("RGB")).unsqueeze(0).to(device)
            with torch.no_grad():
                probs_list.append(torch.softmax(model(t), dim=1)[0].cpu().numpy())
        probs_mat = np.array(probs_list)
        return pd.DataFrame({
            "filename": sub["filename"],
            "cluster_id": sub["near_duplicate_group_id"],
            "y_true": sub["class"],
            "y_pred": [CLASS_NAMES[i] for i in np.argmax(probs_mat, axis=1)],
            "p_Cyst": probs_mat[:, 0],
            "p_Normal": probs_mat[:, 1],
            "p_Stone": probs_mat[:, 2],
            "p_Tumor": probs_mat[:, 3],
        })
        
    val_rn = eval_rn18("validation")
    test_rn = eval_rn18("test")
    eval_rn = generate_full_evaluation(val_rn, test_rn, bootstrap_iterations=BOOTSTRAP_RESAMPLES, seed=42)
    
    save_contract_artifacts(
        out_dir=MODELS_DIR,
        family="resnet18",
        version_tag="resnet18_baseline_v001",
        display_name="ResNet18 Transfer Learning Baseline",
        base_model="resnet18",
        license_name="BSD-3-Clause",
        hyperparameters={"epochs": 20, "lr": 1e-4, "seed": 42, "smoke_test": SMOKE_TEST},
        preprocessing_spec={"resize": [224, 224], "normalization": "ImageNet"},
        library_versions={"torch": torch.__version__},
        training_time_sec=0.0,
        evaluation_results=eval_rn,
        predict_example_code="# Loaded via standard PyTorch ResNet18 pipeline",
        seed=42
    )
    print("Successfully generated ResNet18 common_eval metrics.")
elif not resnet18_ckpt.exists():
    print(f"ResNet18 checkpoint missing at {resnet18_ckpt.as_posix()}! Ensure baseline weights are placed here.")

# Verify all models
families = ["resnet18", "dinov3", "yolo26", "qknn"]
print("=" * 60)
for fam in families:
    m_path = MODELS_DIR / fam / "metrics.json"
    c_path = MODELS_DIR / fam / "model_card.json"
    status = "READY" if m_path.exists() and c_path.exists() else "MISSING (Run prior notebook)"
    print(f"   - {fam.upper():<10}: {status}")
print("=" * 60)

---
### 3. Aggregate Performance Table with 95% Confidence Intervals
* **What it does:** Reads `metrics.json` from each family directory, extracts test metrics and 95% CIs, and formats a comparative benchmark table.
* **Expected runtime:** ~1-2 seconds.
* **Expected output:** Markdown/ASCII table displaying Slice Macro F1, Cluster Macro F1, Stone Recall, and Tumor Recall with CIs.

In [ ]:
summary_rows = []

for fam in ["resnet18", "dinov3", "yolo26", "qknn"]:
    m_path = MODELS_DIR / fam / "metrics.json"
    c_path = MODELS_DIR / fam / "model_card.json"
    if not m_path.exists() or not c_path.exists():
        continue
        
    with open(m_path, "r") as f:
        m = json.load(f)
    with open(c_path, "r") as f:
        card = json.load(f)
        
    test = m["test"]
    ci = test["confidence_intervals_95"]
    
    f1_mean = test["slice_macro_f1"]
    f1_lo = ci["slice_macro_f1"]["ci_lower_95"]
    f1_hi = ci["slice_macro_f1"]["ci_upper_95"]
    
    summary_rows.append({
        "Model Family": fam,
        "Display Name": card.get("display_name", fam),
        "Slice Acc": f"{test['slice_accuracy']:.4f}",
        "Slice Macro F1 [95% CI]": f"{f1_mean:.4f} [{f1_lo:.4f}, {f1_hi:.4f}]",
        "Cluster Macro F1": f"{test['cluster_macro_f1']:.4f}",
        "Stone Recall": f"{test['per_class']['Stone']['recall']:.4f}",
        "Tumor Recall": f"{test['per_class']['Tumor']['recall']:.4f}",
        "_f1_mean": f1_mean,
        "_f1_lo": f1_lo,
        "_f1_hi": f1_hi
    })

df_summary = pd.DataFrame(summary_rows)
display_cols = ["Model Family", "Display Name", "Slice Acc", "Slice Macro F1 [95% CI]", "Cluster Macro F1", "Stone Recall", "Tumor Recall"]
print(df_summary[display_cols].to_markdown(index=False))

---
### 4. Statistical CI Overlap Analysis
* **What it does:** Evaluates pairwise CI overlap between models to report whether observed differences in Macro F1 are statistically indistinguishable or significant.
* **Expected runtime:** < 1 second.
* **Expected output:** Statistical significance report.

In [ ]:
print("=" * 60)
print("STATISTICAL OVERLAP ANALYSIS (95% Cluster-Bootstrap CIs)")
print("=" * 60)

for i in range(len(summary_rows)):
    for j in range(i + 1, len(summary_rows)):
        m1 = summary_rows[i]
        m2 = summary_rows[j]
        
        overlap = not (m1["_f1_hi"] < m2["_f1_lo"] or m2["_f1_hi"] < m1["_f1_lo"])
        
        status_text = "STATISTICALLY INDISTINGUISHABLE (CIs overlap)" if overlap else "STATISTICALLY SIGNIFICANT DIFFERENCE (CIs do not overlap)"
        print(f"{m1['Model Family'].upper()} vs {m2['Model Family'].upper()}:")
        print(f"   {m1['Model Family']}: [{m1['_f1_lo']:.4f}, {m1['_f1_hi']:.4f}] | {m2['Model Family']}: [{m2['_f1_lo']:.4f}, {m2['_f1_hi']:.4f}]")
        print(f"   Result: {status_text}\n")
print("=" * 60)

---
### 5. Generate Comparison Visualizations & `comparison.json`
* **What it does:** Plots comparative Macro F1 chart with error bars and writes the final `comparison.json` report.
* **Expected runtime:** ~2-4 seconds.
* **Expected output:** Saved chart and written `comparison.json`.

In [ ]:
import matplotlib.pyplot as plt

if summary_rows:
    plt.figure(figsize=(10, 5), dpi=150)
    families = [r["Model Family"] for r in summary_rows]
    means = [r["_f1_mean"] for r in summary_rows]
    lowers = [r["_f1_mean"] - r["_f1_lo"] for r in summary_rows]
    uppers = [r["_f1_hi"] - r["_f1_mean"] for r in summary_rows]

    yerr = [lowers, uppers]
    bars = plt.bar(families, means, yerr=yerr, capsize=6, color=["#3b82f6", "#10b981", "#f59e0b", "#8b5cf6"][:len(families)], edgecolor="black", alpha=0.85)

    for bar, mean in zip(bars, means):
        plt.text(bar.get_x() + bar.get_width()/2, mean / 2, f"{mean:.4f}", ha='center', va='center', color='white', fontweight='bold', fontsize=11)

    plt.title("StoneSense-AI: CT Model Comparison — Test Macro F1 (95% Cluster-Bootstrap CIs)", fontsize=12, fontweight="bold")
    plt.ylabel("Macro F1 Score")
    plt.ylim(0.0, 1.05)
    plt.grid(axis="y", linestyle="--", alpha=0.5)

    chart_path = MODELS_DIR / "model_comparison_f1.png"
    plt.savefig(chart_path, bbox_inches="tight")
    plt.close()
    print(f"Saved comparison chart to {chart_path.as_posix()}")

    comparison_data = {
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "models_evaluated": families,
        "summary_table": summary_rows,
        "charts": [chart_path.name]
    }

    with open(COMPARISON_OUT, "w", encoding="utf-8") as f:
        json.dump(comparison_data, f, indent=2)

    print(f"Written comparison report to {COMPARISON_OUT.as_posix()}")
else:
    print("No evaluated models available yet to plot comparison chart.")

---
### 6. Final Deployment Packaging
* **What it does:** Packages all CT models and comparison report into `ct_multimodel_bundle.zip` for production deployment.
* **Expected runtime:** ~3-5 seconds.
* **Expected output:** Final archive summary.

In [ ]:
import shutil

final_zip = WORKSPACE_ROOT / "dl" / "models" / "ct" / "ct_multimodel_bundle.zip"
shutil.make_archive(str(final_zip.with_suffix("")), 'zip', MODELS_DIR)

print("=" * 60)
print("MULTI-MODEL CT SUITE COMPLETE!")
print(f"Final Archive: {final_zip.name}")
print(f"Contains models: {families}")
print("=" * 60)